In [16]:
# import libraries 
import pandas as pd
import numpy as np
import json 
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder


### Load the train, validation, and test splits from the notebook 3

In [17]:
train_df=pd.read_csv(r"/media/lujain/Yeni Birim/Olist/train.csv")
val_df=pd.read_csv(r"/media/lujain/Yeni Birim/Olist/validation.csv")
test_df=pd.read_csv(r"/media/lujain/Yeni Birim/Olist/test.csv")


print(f"Train shape:      {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape:       {test_df.shape}")

Train shape:      (79552, 24)
Validation shape: (9944, 24)
Test shape:       (9945, 24)


### Drop columns that are not available at prediction time

In [18]:
# Drop Leakage & High-Cardinality ID Columns
leak_cols=[
      "order_status",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "avg_review_score",
]
# Identifier columns that don't generalize as features
id_cols=["order_id","customer_id","customer_unique_id","seller_id"]

drop_cols=leak_cols+id_cols

train_df=train_df.drop(columns=drop_cols)
val_df=val_df.drop(columns=drop_cols)
test_df=test_df.drop(columns=drop_cols)

print(f"Dropped {len(drop_cols)} columns.Columns remaining: {train_df.shape[1]}")


Dropped 9 columns.Columns remaining: 15


### Extract Time-Based Features

In [19]:
def add_time_features(df):
    df=df.copy()
    df["order_purchase_timestamp"]=pd.to_datetime(df["order_purchase_timestamp"])
    df["order_estimated_delivery_date"]=pd.to_datetime(df["order_estimated_delivery_date"])

    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour
    df["is_weekend"] = df["purchase_dayofweek"].isin([5, 6]).astype(int)

    # Days  Olist promised at checkout
    df["estimated_delivery_days"] = (
            df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
        ).dt.days
    
    return df.drop(columns=["order_purchase_timestamp", "order_estimated_delivery_date"])


train_df = add_time_features(train_df)
val_df   = add_time_features(val_df)
test_df  = add_time_features(test_df)

train_df[["purchase_month", "purchase_dayofweek", "purchase_hour",
          "is_weekend", "estimated_delivery_days"]].head()

,purchase_month,purchase_dayofweek,purchase_hour,is_weekend,estimated_delivery_days
0,9,6,21,1,45
1,9,0,0,0,52
2,9,1,15,0,16
3,9,3,12,0,18
4,10,6,22,1,22


In [20]:
def add_zip_region(df):
    df=df.copy()
    df["zip_region"]=df["customer_zip_code_prefix"].astype(str).str[0]
    return df.drop(columns=["customer_zip_code_prefix"])

train_df=add_zip_region(train_df)
val_df=add_zip_region(val_df)
test_df=add_zip_region(test_df)

train_df["zip_region"].value_counts()

zip_region
1    15367
2    13892
3    11576
8     8819
9     6642
7     6434
4     6400
6     5473
5     4949
Name: count, dtype: int64

### Frequency encode high cardinality categoricals


In [21]:
def fit_frequency_map(series):
    return series.value_counts(normalize=True).to_dict()
def apply_frequency_map(series,freq_map):
    return series.map(freq_map).fillna(0.0)
freq_encoders={}
for col in ["customer_city"]:
    freq_encoders[col]=fit_frequency_map(train_df[col])

    train_df[f"{col}_freq"]=apply_frequency_map(train_df[col],freq_encoders[col])
    val_df[f"{col}_freq"]=apply_frequency_map(val_df[col],freq_encoders[col])
    test_df[f"{col}_freq"]=apply_frequency_map(test_df[col],freq_encoders[col])

    train_df=train_df.drop(columns=[col])
    val_df=val_df.drop(columns=[col])
    test_df=test_df.drop(columns=[col])
train_df["customer_city_freq"].describe()

count    79552.000000
mean         0.030826
std          0.052978
min          0.000013
25%          0.000553
50%          0.002866
75%          0.020829
max          0.149751
Name: customer_city_freq, dtype: float64

### separate the features and target after cleaning

In [22]:
target_col="is_late"

X_train, y_train=train_df.drop(columns=[target_col]),train_df[target_col]
X_val,y_val=val_df.drop(columns=[target_col]),val_df[target_col]
X_test,y_test=test_df.drop(columns=[target_col]),test_df[target_col]

X_train.dtypes

customer_state                 str
total_price                float64
avg_item_price             float64
max_item_price             float64
min_item_price             float64
total_freight              float64
total_items                float64
total_pyment_value         float64
max_installments           float64
primary_pyment_type            str
purchase_month               int32
purchase_dayofweek           int32
purchase_hour                int32
is_weekend                   int64
estimated_delivery_days      int64
zip_region                     str
customer_city_freq         float64
dtype: object

### Define the numeric and categorical feature groups

In [23]:
numeric_features = [
    "total_price", "avg_item_price", "max_item_price", "min_item_price",
    "total_freight", "total_items", "total_pyment_value", "max_installments",
    "purchase_month", "purchase_dayofweek", "purchase_hour", "is_weekend",
    "estimated_delivery_days", "customer_city_freq", 
]
categorical_features = ["customer_state", "primary_pyment_type", "zip_region"]

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")


Numeric features (14): ['total_price', 'avg_item_price', 'max_item_price', 'min_item_price', 'total_freight', 'total_items', 'total_pyment_value', 'max_installments', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'is_weekend', 'estimated_delivery_days', 'customer_city_freq']
Categorical features (3): ['customer_state', 'primary_pyment_type', 'zip_region']


### Build and fit the preprocessing pipeline

In [24]:
numeric_pipeline=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="median",add_indicator=True)),
    ("scaler",StandardScaler()),
])
categorical_pipeline=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore")),
])
preprocessor=ColumnTransformer(transformers=[
    ("num",numeric_pipeline,numeric_features),
    ("cat",categorical_pipeline,categorical_features),
])
# The train only
X_train_processed=preprocessor.fit_transform(X_train)

# Transform for validation and test
X_val_processed=preprocessor.transform(X_val)
X_test_processed=preprocessor.transform(X_test)

print(f"Train processed shape:      {X_train_processed.shape}")
print(f"Validation processed shape: {X_val_processed.shape}")
print(f"Test processed shape:       {X_test_processed.shape}")


Train processed shape:      (79552, 62)
Validation processed shape: (9944, 62)
Test processed shape:       (9945, 62)


### Turn the transformed arrays back into labeled feature tables


In [25]:
feature_names=preprocessor.get_feature_names_out().tolist()
X_train_df=pd.DataFrame(X_train_processed,columns=feature_names,index=X_train.index)
X_val_df=pd.DataFrame(X_val_processed,columns=feature_names,index=X_val.index)
X_test_df=pd.DataFrame(X_test_processed,columns=feature_names,index=X_test.index)

X_train_df.head()

,num__total_price,num__avg_item_price,num__max_item_price,num__min_item_price,num__total_freight,num__total_items,num__total_pyment_value,num__max_installments,num__purchase_month,num__purchase_dayofweek,...,cat__primary_pyment_type_voucher,cat__zip_region_1,cat__zip_region_2,cat__zip_region_3,cat__zip_region_4,cat__zip_region_5,cat__zip_region_6,cat__zip_region_7,cat__zip_region_8,cat__zip_region_9
0,-0.310605,-0.473647,-0.456714,-0.489722,2.014080,1.580468,-0.109274,-0.715425,0.898925,1.649115,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,-0.375592,-0.350408,-0.352774,-0.347474,-0.330091,-0.262126,-0.390880,0.014755,0.898925,-1.405637,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.242609,-0.246172,-0.244678,-0.243193,-0.269745,-0.262126,-0.547912,-0.350335,0.898925,-0.896511,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-0.009307,-0.427970,-0.430077,-0.425069,-0.676957,3.423062,-0.253737,-0.350335,0.898925,0.121739,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.179030,-0.133918,-0.137008,-0.130892,-0.635255,-0.262126,-0.233067,-0.715425,1.180539,1.649115,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Check if feature table counts match the leble counts

In [26]:
print("Remaining NaNs per split:")
print(f"Train:       {X_train_df.isnull().sum().sum()}")
print(f"Validation:  {X_val_df.isnull().sum().sum()}")
print(f"Test:        {X_test_df.isnull().sum().sum()}")

Remaining NaNs per split:
Train:       0
Validation:  0
Test:        0


### Artifact: the final feature table, the fitted transformers, and the feature list

In [27]:
# save the fitted transformers
joblib.dump(preprocessor,"preprocessor.joblib")
joblib.dump(freq_encoders,"freq_encoders.joblib")

# save the feature list
with open("feature_list.json","w")as f:
    json.dump(feature_names,f,indent=2)

# save the feature tables
X_train_df.to_csv("train_feature.csv",index=False)
X_val_df.to_csv("validation_feature.csv",index=False)
X_test_df.to_csv("test_feature.csv",index=False)

# save the labels separately from the features
y_train.to_csv("train_labels.csv",index=False)
y_val.to_csv("validation_labels.csv",index=False)
y_test.to_csv("test_labels.csv",index=False)

print("Artifacts saved successfully:")
print("  preprocessor.joblib, freq_encoders.joblib, feature_list.json")
print("  train_features.csv, validation_features.csv, test_features.csv")
print("  train_labels.csv, validation_labels.csv, test_labels.csv")


Artifacts saved successfully:
  preprocessor.joblib, freq_encoders.joblib, feature_list.json
  train_features.csv, validation_features.csv, test_features.csv
  train_labels.csv, validation_labels.csv, test_labels.csv
